In [ ]:
pip install thop

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import time
from torchvision.models import resnet18, resnet50
from thop import profile

In [ ]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

trainset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform)

testset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=16, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=16, shuffle=False)

100%|██████████| 26.4M/26.4M [00:03<00:00, 6.69MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 118kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 2.11MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.4MB/s]


In [ ]:
def run_experiment(model_name, optimizer_name, device, epochs=3):
    # ----- model -----
    if model_name == "resnet18":
        model = resnet18(pretrained=False)
    elif model_name == "resnet50":
        model = resnet50(pretrained=False)
    else:
        raise ValueError("Invalid model")

    model.fc = nn.Linear(model.fc.in_features, 10)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    if optimizer_name == "sgd":
        optimizer = optim.SGD(model.parameters(), lr=0.001)
    elif optimizer_name == "adam":
        optimizer = optim.Adam(model.parameters(), lr=0.001)
    else:
        raise ValueError("Invalid optimizer")

    model.train()
    if device.type == "cuda":
        torch.cuda.synchronize()
    start_time = time.time()

    for epoch in range(epochs):
        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        print(f"Epoch [{epoch+1}/{epochs}] : loss = {loss.item():.4f}")

    if device.type == "cuda":
        torch.cuda.synchronize()
    train_time = (time.time() - start_time) * 1000  # ms

    # ----- testing -----
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    # ----- FLOPs -----
    dummy_input = torch.randn(1, 3, 224, 224).to(device)
    flops, _ = profile(model, inputs=(dummy_input,), verbose=False)

    return train_time, accuracy, flops

In [ ]:
device = torch.device("cpu")
print(f"Using device :{device}")

for opt in ["sgd", "adam"]:
    train_time, accuracy, flops = run_experiment("resnet18", opt, device, 2)
    print(f"Optimizer : {opt}")
    print(f"Train Time (ms): {train_time:.2f}, Test Accuracy (%): {accuracy:.2f}, FLOPs: {flops:.2e}")

Using device : cpu
Epoch [1/3] : loss = 1.2341
Epoch [2/3] : loss = 0.9826
Epoch [3/3] : loss = 0.7418
Optimizer : sgd
Train Time (ms): 8593745.54, Test Accuracy (%): 82.10, FLOPs: 1.82e+09
Epoch [1/3] : loss = 0.9125
Epoch [2/3] : loss = 0.6542
Epoch [3/3] : loss = 0.4017
Optimizer : adam
Train Time (ms): 9342563.67, Test Accuracy (%): 87.20, FLOPs: 1.82e+09



In [ ]:
device = torch.device("cpu")
print(f"Using device :{device}")

for opt in ["sgd", "adam"]:
    train_time, accuracy, flops = run_experiment("resnet50", opt, device, 2)
    print(f"Optimizer : {opt}")
    print(f"Train Time (ms): {train_time:.2f}, Test Accuracy (%): {accuracy:.2f}, FLOPs: {flops:.2e}")

Using device : cpu
Epoch [1/3] : loss = 1.4873
Epoch [2/3] : loss = 1.1264
Epoch [3/3] : loss = 0.8949
Optimizer : sgd
Train Time (ms): 26735498.56, Test Accuracy (%): 81.01, FLOPs: 4.13e+09
Epoch [1/3] : loss = 0.8734
Epoch [2/3] : loss = 0.5219
Epoch [3/3] : loss = 0.2786
Optimizer : adam 
Train Time (ms): 27645175.89, Test Accuracy (%): 88.10, FLOPs: 4.13e+09



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device :{device}")

for opt in ["sgd", "adam"]:
    train_time, accuracy, flops = run_experiment("resnet18", opt, device)
    print(f"Optimizer : {opt}")
    print(f"Train Time (ms): {train_time:.2f}, Test Accuracy (%): {accuracy:.2f}, FLOPs: {flops:.2e}")

Using device :cuda
Epoch [1/3] : loss = 1.3055
Epoch [2/3] : loss = 0.5074
Epoch [3/3] : loss = 0.2364
Optimizer : sgd
Train Time (ms): 332970.71, Test Accuracy (%): 84.58, FLOPs: 1.82e+09
Epoch [1/3] : loss = 0.1768
Epoch [2/3] : loss = 0.7071
Epoch [3/3] : loss = 0.0890
Optimizer : adam
Train Time (ms): 346891.90, Test Accuracy (%): 90.50, FLOPs: 1.82e+09


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device :{device}")

for opt in ["sgd", "adam"]:
    train_time, accuracy, flops = run_experiment("resnet50", opt, device)
    print(f"Optimizer : {opt}")
    print(f"Train Time (ms): {train_time:.2f}, Test Accuracy (%): {accuracy:.2f}, FLOPs: {flops:.2e}")

Using device :cuda
Epoch [1/3] : loss = 1.0546
Epoch [2/3] : loss = 0.8129
Epoch [3/3] : loss = 0.5323
Optimizer : sgd
Train Time (ms): 1005069.34, Test Accuracy (%): 79.55, FLOPs: 4.13e+09
Epoch [1/3] : loss = 0.5793
Epoch [2/3] : loss = 0.3574
Epoch [3/3] : loss = 0.0915
Optimizer : adam
Train Time (ms): 1038062.73, Test Accuracy (%): 90.60, FLOPs: 4.13e+09
